In [0]:
%python
from datetime import datetime, timezone

booking = {
    "booking_id": "B001",
    "passenger_id": "P001",
    "booking_status": "CONFIRMED",
    "created_at": datetime.now(timezone.utc).isoformat()
}

booking_segment = {
    "booking_segment_id": "BS001",
    "booking_id": "B001",
    "original_flight_leg_id": "FL100_20261115",
    "assigned_flight_leg_id": "FL100_20261115",
    "assignment_status": "ASSIGNED"
}


rebooked_segment = {
    "booking_segment_id": "BS001",
    "booking_id": "B001",
    "original_flight_leg_id": "FL100_20261115",
    "assigned_flight_leg_id": "FL200_20261115",
    "assignment_status": "ASSIGNED"
}
initial_assignment_event = {
    "event_id": "EVT_BS001_V1",
    "entity_id": "BS001",
    "entity_version": 1,
    "effective_at": "2026-11-14T08:00:00Z",
    "payload": booking_segment
}

current_assignment_event = {
    "event_id": "EVT_BS001_V2",
    "entity_id": "BS001",
    "entity_version": 2,
    "effective_at": "2026-11-14T10:00:00Z",
    "payload": rebooked_segment
}

In [0]:
current_assignment_event_V2 = current_assignment_event.copy()
current_assignment_event_V2["payload"]["assigned_flight_leg_id"] = 'FL300_20261115'

incoming_events = [
    current_assignment_event,
    initial_assignment_event,
    current_assignment_event,
    current_assignment_event_V2
]

latest_by_entity = {}

for event in incoming_events:
    entity_id = event["entity_id"]

    if entity_id not in latest_by_entity:
        latest_by_entity[entity_id] = event
    else:
        if event["entity_version"] > latest_by_entity[entity_id]["entity_version"]:
            latest_by_entity[entity_id] = event
for entity_id, event in latest_by_entity.items():
    print(entity_id, event["entity_version"])

print(latest_by_entity)

In [0]:


# latest_by_entity = {}

# for i in incoming_events:
#     entity_id = i['entity_id']

#     if entity_id not in latest_by_entity:
#         latest_by_entity[entity_id] = []
#     else:
#         if event["entity_version"] > latest_by_entity[entity_id]["entity_version"]:
#             latest_by_entity[entity_id] = event

# for entity_id, events in latest_by_entity.items():
#     for event in events:
#         print(entity_id, event['entity_version'])

In [0]:
from copy import deepcopy

current_assignment_event_V2 = deepcopy(current_assignment_event)
current_assignment_event_V2["payload"]["assigned_flight_leg_id"] = "FL300_20261115"


incoming_events = [
    current_assignment_event,
    initial_assignment_event,
    current_assignment_event,
    current_assignment_event_V2
]

latest_by_entity = {}
seen_by_event_id = {}

for event in incoming_events:
    event_id = event["event_id"]

    # Check event_id first
    if event_id in seen_by_event_id:

        # Retry: same ID and same contents
        if seen_by_event_id[event_id] == event:
            continue
        else:    
            raise ValueError(f"Conflicting event_id: {event_id}")      # Conflict: same ID but different contents

    # Remember event_id
    seen_by_event_id[event_id] = event

    # Then do version selection
    entity_id = event["entity_id"]

    if entity_id not in latest_by_entity:
        latest_by_entity[entity_id] = event

    elif event["entity_version"] > latest_by_entity[entity_id]["entity_version"]:
        latest_by_entity[entity_id] = event

for entity_id, event in latest_by_entity.items():
    print(entity_id, event["entity_version"])

In [0]:
latest_by_entity = {}
seen_by_event_id = {}

try:
    for event in incoming_events:
        event_id = event["event_id"]

        # Deduplication / conflict detection
        if event_id in seen_by_event_id:

            if seen_by_event_id[event_id] == event:
                continue

            raise ValueError(f"Conflicting event_id: {event_id}")

        seen_by_event_id[event_id] = event

        # Latest version selection
        entity_id = event["entity_id"]

        if entity_id not in latest_by_entity:
            latest_by_entity[entity_id] = event

        elif event["entity_version"] > latest_by_entity[entity_id]["entity_version"]:
            latest_by_entity[entity_id] = event

except ValueError as e:
    print("Error:", e)
    print("\nCurrent latest_by_entity:")
    for entity_id, event in latest_by_entity.items():
        print(entity_id, event["entity_version"])